# 🎯 Few-Shot Open-Set KWS — Training on Colab

**Pipeline**: Setup → Data → MFCC → DSCNN-L → Triplet Loss → Checkpoints

**GPU**: Runtime → Change runtime type → T4 GPU

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/DoAnTotNghiep'
!mkdir -p {DRIVE_PROJECT}/checkpoints {DRIVE_PROJECT}/results

In [ ]:
# Upload file RAR/ZIP từ máy
from google.colab import files
uploaded = files.upload()

In [ ]:
import os, sys

# Giải nén RAR/ZIP
for f in os.listdir('/content/'):
    if f.endswith('.rar'):
        !apt-get install -qq unrar
        !unrar x -o+ /content/{f} /content/
    elif f.endswith('.zip') and 'DoAnTotNghiep' in f:
        !unzip -qo /content/{f} -d /content/

# CD vào project
os.chdir('/content/DoAnTotNghiep')
sys.path.insert(0, '/content/DoAnTotNghiep')

!pip install -q torch torchaudio numpy pyyaml matplotlib tensorboard scikit-learn soundfile requests tqdm

import torch
print(f'Project: {os.getcwd()}')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Datasets

In [ ]:
%%time
# Download Google Speech Commands v2 (~2.3GB)
!python data/download_gsc.py

In [ ]:
%%time
# Download DEMAND noise (16kHz only)
import zipfile, requests
from pathlib import Path
from tqdm import tqdm

DEMAND_DIR = Path('data/demand')
DEMAND_DIR.mkdir(parents=True, exist_ok=True)

if not any(DEMAND_DIR.rglob('*.wav')):
    BASE = 'https://zenodo.org/records/1227121/files'
    ENVS = [
        'DKITCHEN', 'DLIVING', 'DWASHING',
        'NFIELD', 'NPARK', 'NRIVER',
        'OHALLWAY', 'OMEETING', 'OOFFICE',
        'PCAFETER', 'PRESTO', 'PSTATION',
        'SPSQUARE', 'STRAFFIC',
        'TBUS', 'TCAR', 'TMETRO',
    ]
    for env in tqdm(ENVS, desc='Downloading DEMAND'):
        url = f'{BASE}/{env}_16k.zip?download=1'
        zip_path = Path(f'data/{env}_16k.zip')
        if not zip_path.exists():
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DEMAND_DIR)
        zip_path.unlink()

print(f'DEMAND: {len(list(DEMAND_DIR.rglob("*.wav")))} noise files')

## 3. Build Dataset & DataLoader

In [ ]:
import random, yaml
import numpy as np
import torch
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from src.features.mfcc import MFCCExtractor
from src.features.augmentation import NoiseAugmenter
from src.models.dscnn import DSCNN
from src.models.prototypical import EpisodicBatchSampler, TripletLoss, train_one_epoch

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config:', list(cfg.keys()))

In [ ]:
class KWSAudioDataset(Dataset):
    """Load WAV -> pad/trim 1s -> MFCC -> optional noise augmentation."""

    def __init__(self, data_dir, words, mfcc_extractor, augmenter=None, max_samples_per_word=200):
        self.mfcc = mfcc_extractor
        self.augmenter = augmenter
        self.files, self.labels, self.word_to_idx = [], [], {}

        for i, word in enumerate(sorted(words)):
            self.word_to_idx[word] = i
            word_dir = Path(data_dir) / word
            if not word_dir.exists():
                print(f'  Warning: {word_dir} not found')
                continue
            for wav in sorted(word_dir.glob('*.wav'))[:max_samples_per_word]:
                self.files.append(wav)
                self.labels.append(i)

        print(f'Dataset: {len(self.files)} samples, {len(set(self.labels))} classes')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.files[idx])
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if self.augmenter is not None:
            waveform = self.augmenter.augment(waveform)
        return self.mfcc.extract(waveform), self.labels[idx]

In [ ]:
# MFCC Extractor
mfcc_extractor = MFCCExtractor(n_mfcc=cfg['mfcc']['n_mfcc'], num_features=cfg['mfcc']['num_features'])

# Noise Augmenter
demand_dir = Path(cfg['noise']['demand_dir'])
if demand_dir.exists() and any(demand_dir.rglob('*.wav')):
    augmenter = NoiseAugmenter(demand_dir, prob=cfg['noise']['prob'], snr_db=cfg['noise']['snr_db'])
    print(f'Augmenter: {len(augmenter.noise_files)} noise files, p={augmenter.prob}, SNR={augmenter.snr_db}dB')
else:
    augmenter = None
    print('No DEMAND noise — training without augmentation')

# Verify
test_mfcc = mfcc_extractor.extract(torch.randn(1, 16000))
print(f'MFCC: {test_mfcc.shape}')  # (1, 47, 10)
assert test_mfcc.shape == (1, 47, 10)

In [ ]:
# Build dataset
GSC_DIR = Path(cfg['data']['gsc_dir'])
EXCLUDED = {'backward', 'forward', 'visual', 'follow', 'learn'}
all_words = sorted([d.name for d in GSC_DIR.iterdir() if d.is_dir() and not d.name.startswith('_')])
train_words = [w for w in all_words if w not in EXCLUDED]
print(f'Words ({len(train_words)}): {train_words}')

train_dataset = KWSAudioDataset(GSC_DIR, train_words, mfcc_extractor, augmenter, max_samples_per_word=200)

# Episodic sampler
n_cls = min(cfg['training']['n_classes'], len(set(train_dataset.labels)))
n_samp = cfg['training']['n_samples']
n_ep = cfg['training']['episodes_per_epoch']
print(f'Episodic: {n_cls} classes x {n_samp} samples x {n_ep} episodes')

sampler = EpisodicBatchSampler(train_dataset.labels, n_cls, n_samp, n_ep)
train_loader = DataLoader(train_dataset, batch_sampler=sampler, num_workers=2, pin_memory=True)

## 4. Model & Optimizer

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(cfg['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = DSCNN(model_size=cfg['model']['architecture'][-1]).to(device)
print(f'DSCNN-{cfg["model"]["architecture"][-1]}: {sum(p.numel() for p in encoder.parameters()):,} params')

with torch.no_grad():
    out = encoder(torch.randn(2, 1, 47, 10).to(device))
    print(f'Output: {out.shape}')  # (2, 276)
    assert out.shape == (2, 276)

In [ ]:
tc = cfg['training']
optimizer = torch.optim.Adam(encoder.parameters(), lr=tc['optimizer']['lr'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=tc['scheduler']['step_size'], gamma=tc['scheduler']['gamma'])
loss_fn = TripletLoss(margin=tc['triplet_margin'])

print(f"Adam lr={tc['optimizer']['lr']}, TripletLoss margin={tc['triplet_margin']}")
print(f"Plan: {tc['epochs']} epochs x {tc['episodes_per_epoch']} episodes")

## 5. Training Loop

In [ ]:
import time
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

CKPT_DIR = Path(DRIVE_PROJECT) / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter('runs/kws_training')

NUM_EPOCHS = tc['epochs']
SAVE_EVERY = cfg['checkpoint']['save_every']
best_loss = float('inf')

print(f'Training {NUM_EPOCHS} epochs, save every {SAVE_EVERY}')
print(f'Checkpoints: {CKPT_DIR}')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    metrics = train_one_epoch(encoder, train_loader, optimizer, loss_fn, device)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    writer.add_scalar('Loss/train', metrics['loss'], epoch)
    writer.add_scalar('LR', lr, epoch)
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {metrics['loss']:.4f} | LR: {lr:.6f} | Time: {time.time()-t0:.1f}s")

    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
        ckpt = {
            'epoch': epoch,
            'model_state_dict': encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': metrics['loss'],
        }
        torch.save(ckpt, CKPT_DIR / 'latest.pt')
        if metrics['loss'] < best_loss:
            best_loss = metrics['loss']
            torch.save(ckpt, CKPT_DIR / 'best.pt')
            print(f'  ★ New best: {best_loss:.4f}')
        torch.save(ckpt, CKPT_DIR / f'epoch_{epoch+1:02d}.pt')

writer.close()
print(f'\n✅ Done! Best loss: {best_loss:.4f}')

## 6. Verification

In [ ]:
# Load best checkpoint & test with real GSC audio
best_ckpt = torch.load(CKPT_DIR / 'best.pt', map_location=device, weights_only=False)
encoder.load_state_dict(best_ckpt['model_state_dict'])
encoder.eval()
print(f"Best checkpoint: epoch {best_ckpt['epoch']+1}, loss {best_ckpt['loss']:.4f}")

test_words = ['yes', 'no', 'stop']
protos = {}
for word in test_words:
    word_dir = GSC_DIR / word
    if not word_dir.exists(): continue
    wavs = sorted(word_dir.glob('*.wav'))[:5]  # 5-shot
    embs = []
    for wav in wavs:
        w, sr = torchaudio.load(wav)
        if sr != 16000: w = torchaudio.transforms.Resample(sr, 16000)(w)
        m = mfcc_extractor.extract(w).unsqueeze(0).to(device)
        with torch.no_grad():
            embs.append(F.normalize(encoder(m), p=2, dim=-1).squeeze(0))
    protos[word] = torch.stack(embs).mean(dim=0)
    print(f'{word}: prototype from {len(embs)} samples')

# Inter-prototype distances (higher = better separation)
words = list(protos.keys())
for i, w1 in enumerate(words):
    for w2 in words[i+1:]:
        d = F.pairwise_distance(protos[w1].unsqueeze(0), protos[w2].unsqueeze(0)).item()
        print(f'  d({w1}, {w2}) = {d:.4f}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 7. Copy Results to Drive

In [ ]:
import shutil
drive_logs = Path(DRIVE_PROJECT) / 'runs'
if Path('runs/kws_training').exists():
    if drive_logs.exists(): shutil.rmtree(drive_logs)
    shutil.copytree('runs', drive_logs)
    print(f'Logs copied to {drive_logs}')

print(f'\nCheckpoints:')
for f in sorted(CKPT_DIR.glob('*.pt')):
    print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f}MB')